# Lecture 14 - Arrays, linked lists and hash tables

**Lecture Topic**: The storage underneath the costs. Why indexing is instant, why growing an array in a loop is slow, and the hash table that pandas runs on.

## Announcements

- Module C continues: last time we measured *how* code scales, today we look at *why*, through the data structures underneath.
- Keep going on your project work.
- ClassBuzz poll for today: `JXFM`

## Recap: last time

- Complexity is about how the work grows with the input size.
- We measured it: the slope on log-log axes is the exponent.
- Two results we left unexplained:
  - a `set` membership test was effectively **O(1)** per item,
  - growing a NumPy array with `np.append` in a loop was **O(n^2)**.

Today is the "why": arrays, linked lists, and hash tables, and how the choice of structure sets those costs.

Today:
 - Contiguous storage (the array) and amortized append
 - Linked storage (the linked list)
 - Hash tables: O(1) membership
 - Real implementations: pandas indexes, group-by, and joins are hash tables

# Contiguous storage: the array

A Python list and a NumPy array both store their elements in one contiguous block of memory. That one fact explains most of their costs.

 - **Indexing is O(1).** The element at position `k` lives at `start + k * itemsize`, so getting `a[k]` is a single address calculation, no searching.
 - **Growing is the catch.** The block has a fixed size. To make it bigger you allocate a new, larger block and copy everything across.

Python lists hide that cost with a trick: they over-allocate, leaving spare room at the end. Most appends drop into spare room and cost O(1); only occasionally does the list have to grow and copy. Averaged over many appends, each one is **amortized O(1)**.

Inserting at the *front* gets no such deal: every element has to shift up one place, so it is O(n).

In [ ]:
import time

def append_end(n):
    xs = []
    for i in range(n):
        xs.append(i)          # amortized O(1)
    return xs

def insert_front(n):
    xs = []
    for i in range(n):
        xs.insert(0, i)       # shifts everything up: O(n) each time
    return xs

for f in [append_end, insert_front]:
    start = time.perf_counter()
    f(160_000)
    print(f"{f.__name__:13s}: {time.perf_counter() - start:.3f} s")

Same number of items, very different cost. `append` rides the amortized O(1); `insert(0, ...)` pays O(n) every time, so the loop is O(n^2). This is the same reason `np.append` in a loop was quadratic last time: a NumPy array does not over-allocate, so *every* append copies the whole thing.

The habit that follows: build with list `append` (or collect the pieces) and convert to an array once, at the end.

### Question 1

You have a Python list of `n` items. Which operation is **not** O(1)?

 - A. Reading `xs[k]` for a known index `k`
 - B. Appending to the end, `xs.append(v)`
 - C. Inserting at the front, `xs.insert(0, v)`
 - D. Reading the last item, `xs[-1]`

**C.** Inserting at the front has to shift all `n` existing elements up one slot, so it is O(n). The other three are constant-time because contiguous storage makes position arithmetic free, and append is amortized O(1).

## The same trap in pandas

The commonest place this bites in real code is growing a DataFrame with `pd.concat` inside a loop. A NumPy array and a DataFrame both store their data in one block, so each `concat` copies the whole result so far. A loop that looks harmless is quietly O(n^2).

In [ ]:
import pandas as pd
import time

taxi = pd.read_csv("../data/2019_Yellow_Taxi_Trip_Data.csv")
chunks = [taxi.iloc[i:i + 2] for i in range(0, len(taxi), 2)]   # ~5000 tiny chunks

def concat_in_loop(chunks):
    out = pd.DataFrame()
    for c in chunks:
        out = pd.concat([out, c])      # copies the growing result every time: O(n^2)
    return out

def concat_once(chunks):
    return pd.concat(chunks)           # collect first, concatenate once: O(n)

for f in [concat_in_loop, concat_once]:
    start = time.perf_counter()
    f(chunks)
    print(f"{f.__name__:14s}: {time.perf_counter() - start:.3f} s")

Same result, but the loop is several times slower, and the gap grows with the data because each `concat` copies everything collected so far. Building a list of the pieces and calling `pd.concat` once at the end does the work in a single pass. This is the pandas version of "build with append, convert once", and it is one of the most common quiet slowdowns in agent-written data code.

# Linked storage: the linked list

A linked list stores each element in its own little object, a **node**, that also holds a reference to the next node. The list itself is just a reference to the first node, the `head`.

Nothing is contiguous, so the trade-offs flip:

 - **Growing and shrinking is cheap.** Adding a node at the front is just making it point at the old head. No block to resize, no copying.
 - **Indexing is O(n).** There is no address arithmetic. To reach item `k` you start at the head and follow `k` references.

In [ ]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None        # reference to the next node, or None at the end

class LinkedList:
    def __init__(self):
        self.head = None

    def add_front(self, value):       # O(1): point the new node at the old head
        node = Node(value)
        node.next = self.head
        self.head = node

    def add_back(self, value):        # O(n): walk to the end first
        node = Node(value)
        if self.head is None:
            self.head = node
            return
        cur = self.head
        while cur.next is not None:
            cur = cur.next
        cur.next = node

    def to_list(self):                # O(n): follow the chain
        out, cur = [], self.head
        while cur is not None:
            out.append(cur.value)
            cur = cur.next
        return out

In [ ]:
ll = LinkedList()
for v in [1, 2, 3]:
    ll.add_front(v)      # each is O(1)
print("after three add_front:", ll.to_list())
ll.add_back(99)          # this one walked the whole list
print("after add_back:   ", ll.to_list())

### Question 2

In the `LinkedList` above, `add_front` is O(1) but `add_back` is O(n). What single change makes `add_back` O(1) as well?

 - A. Nothing can make it O(1)
 - B. Keep a `tail` reference to the last node, updated on every add
 - C. Store the values in a Python list instead
 - D. Make `add_back` recursive

**B.** If the list also keeps a `tail` reference, `add_back` can jump straight to the end instead of walking there. (Note this helps *adding* at the back; removing from the back of a singly linked list is still O(n), because you need the node *before* the tail and the only way to it is from the head.)

## Array vs. linked list

| Operation | Array / Python list | Linked list |
|---|---|---|
| Index item `k` | **O(1)** | O(n) |
| Insert / remove at front | O(n) | **O(1)** |
| Append at end | O(1) amortized | O(1) with a tail, else O(n) |
| Memory overhead | low (just the data) | higher (a reference per node) |

There is no "best" structure, only the right one for the access pattern. In data science the contiguous array usually wins, because we scan and index far more than we splice, and contiguous memory is what lets NumPy and pandas run their loops in fast compiled C.

# Hash tables: O(1) membership

Back to last time's mystery: how was `x in seen` effectively O(1), when a list membership test `x in some_list` has to scan?

A `set` (and a `dict`) is a **hash table**. A *hash function* turns a key into a number, which picks a bucket to store the key in. To check membership you hash the key, go straight to its bucket, and look only there. No scan, so lookup, insert, and membership are **average O(1)**, independent of how many items are stored.

In [ ]:
# hashing is just a function from a key to an integer
for key in ["taxi", "zone", (36.1, -115.2)]:
    print(f"hash({key!r}) = {hash(key)}")

The catch is **hashability**: a key has to be hashable, which in practice means immutable. If a key could change after it was filed away, its hash would no longer point to the right bucket. So tuples work as keys, but lists do not.

In [ ]:
d = {}
d[(36.1, -115.2)] = "a point is fine: tuples are hashable"
print(d[(36.1, -115.2)])

try:
    d[[36.1, -115.2]] = "a list is not"
except TypeError as e:
    print("TypeError:", e)

### Question 3

Which of these can be used as a **dictionary key**?

 - A. A list `[1, 2]`
 - B. A tuple `(1, 2)`
 - C. A set `{1, 2}`
 - D. A DataFrame

**B, the tuple.** It is immutable, so it is hashable. Lists, sets, and DataFrames are mutable and cannot be keys. This is exactly why you reach for a tuple when you want to key something by a pair like `(zone, hour)`.

(The small print: hashing has a worst case. If many keys collide into the same bucket, lookups degrade toward O(n). Python's hashing makes that rare in practice, so we treat dict and set operations as O(1).)

### Question 4

A real data-cleaning task: find how many trip records are exact duplicates of an earlier one. The two versions below are identical except for how they remember the trips seen so far, a **list** or a **set**. How much faster is the set version?

 - A. About the same
 - B. A little faster (around 2x)
 - C. Many times faster
 - D. The list version wins

In [ ]:
import pandas as pd
import time

taxi = pd.read_csv("../data/2019_Yellow_Taxi_Trip_Data.csv")
trips = list(zip(taxi["tpep_pickup_datetime"], taxi["pulocationid"], taxi["dolocationid"]))

def dups_with_list(trips):
    seen, dups = [], 0
    for t in trips:
        if t in seen:          # scans the growing list: O(k) each time
            dups += 1
        else:
            seen.append(t)
    return dups

def dups_with_set(trips):
    seen, dups = set(), 0
    for t in trips:
        if t in seen:          # hash lookup: O(1)
            dups += 1
        else:
            seen.add(t)
    return dups

def timed(f, data):
    start = time.perf_counter()
    d = f(data)
    return d, time.perf_counter() - start

for N in [5000, 10000]:                       # time the list version at two sizes
    d, t = timed(dups_with_list, trips[:N])
    print(f"list, N={N:>5}: {t:.3f} s  ({d} duplicate trips)")
d, t = timed(dups_with_set, trips)
print(f"set,  N=10000: {t:.3f} s  ({d} duplicate trips)")

**C, many times faster.** Both find the same duplicates. The shape is the real lesson: doubling the data from 5,000 to 10,000 rows roughly quadruples the list version's time, which is the O(n^2) signature from last lecture, because it rescans a growing list of seen trips on every row. The set version does an O(1) hash lookup each time, so it stays linear and finishes too fast to measure. A membership test against a growing list inside a loop is one of the most common quiet quadratics in agent-written code, and swapping the list for a set fixes it.

# Implementations: pandas runs on hash tables

This has important practical applications. The operations you use in pandas every day are hash tables underneath.

A pandas **Index** is a hash table. Once you set an index, label lookups with `.loc` are O(1), not a scan of the column.

In [ ]:
import numpy as np
import pandas as pd
import time

n = 1_000_000
df = pd.DataFrame({"id": np.arange(n), "value": np.random.rand(n)})
keyed = df.set_index("id")                 # builds a hash table on 'id'

lookups = np.random.randint(0, n, size=2000)

start = time.perf_counter()
for k in lookups:
    _ = keyed.loc[k, "value"]              # O(1) hash lookup
t_index = time.perf_counter() - start

start = time.perf_counter()
for k in lookups[:50]:                     # only 50; a full scan each time is slow
    _ = df.loc[df["id"] == k, "value"]     # O(n) boolean scan
t_scan = time.perf_counter() - start

print(f"2000 indexed .loc lookups:   {t_index:.3f} s")
print(f"  50 boolean-mask lookups:   {t_scan:.3f} s")
print(f"per-lookup, the index is about {(t_scan / 50) / (t_index / 2000):.0f}x faster")

`group_by` hashes the group keys to bucket the rows; `merge` hashes the join keys to find matches. It is the same structure every time, and it is why setting a sensible index before repeated lookups or joins is one of the cheapest speed-ups available.

## Many-to-many joins: the blow-up

One hazard worth seeing before it bites you in the project. A `merge` pairs up *every* matching row on the left with *every* matching row on the right. If the key is not unique on both sides, the output is the product of the matches, and the row count can explode.

In [ ]:
import pandas as pd

left = pd.DataFrame({"key": [1, 1, 2], "L": ["a", "b", "c"]})
right = pd.DataFrame({"key": [1, 1, 2], "R": ["x", "y", "z"]})

merged = pd.merge(left, right, on="key")
print(f"left: {len(left)} rows, right: {len(right)} rows")
print(f"merged: {len(merged)} rows")      # key=1 gives 2x2=4, key=2 gives 1 -> 5
print(merged)

Three rows joined to three rows gave five, because the two `key = 1` rows on each side produced four combinations. On real data a duplicated key can turn a million rows into tens of millions and quietly exhaust memory.

The guard is to say what you expect and let pandas check it:

In [ ]:
try:
    pd.merge(left, right, on="key", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", e)

`validate="one_to_one"` (or `"one_to_many"`, `"many_to_one"`) turns a silent blow-up into a loud error, the same lesson as the row-count and `merge` checks from Module A. When an agent writes a join, this is the first thing to pin down.

## Two ways to match rows: hash join and sort-merge join

When you `merge`, pandas has two ways to line the keys up.

 - **Hash join:** build a hash table on one side's key, then probe it with the other side's keys. Around O(n + m), and the usual choice for an unsorted merge.
 - **Sort-merge join:** sort both sides by the key, then walk them together in a single pass. Around O(n log n + m log m), and if a side is already sorted that part of the cost goes away.

So a key you will join on again and again is worth sorting or indexing once. It is the same reason a database keeps a sorted B-tree index on its join columns, which we come back to in the trees lecture.

# Recap

 - An **array** is contiguous: O(1) indexing, but growing means copying (hence amortized append, and the quadratic `np.append` loop).
 - A **linked list** is nodes with references: O(1) at the front, but O(n) to index.
 - A **hash table** (dict, set) gives average **O(1)** membership, insert, and lookup, as long as the keys are hashable.
 - pandas **indexes, group-by, and joins are hash tables**; set an index before repeated lookups, and `validate=` your merges to catch the many-to-many blow-up.

**Next time (S17): stacks, queues and recursion** - two structures built on these, the call stack behind every traceback, and when recursion needs to become a loop.